In [1]:
import pandas as pd

In [2]:
%pip install transformers datasets scikit-learn torch accelerate

In [3]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# ──────────────────────────────────────────────
# 1. CONFIG
# ──────────────────────────────────────────────
MODEL_NAME   = "prajjwal1/bert-tiny"
NUM_LABELS   = 8          # ← change to your number of classes
MAX_LEN      = 512        # max token length
BATCH_SIZE   = 32
EPOCHS       = 20
LR           = 5e-5
WARMUP_RATIO = 0.1        # fraction of steps used for LR warm-up
WEIGHT_DECAY = 0.01
SEED         = 42
SAVE_DIR     = "./bert_tiny_classifier"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

torch.manual_seed(SEED)
np.random.seed(SEED)

#local_dataset_path = "data/conexao-labels-full-4.1nano-p2.csv"
#online_dataset_path = "https://media.githubusercontent.com/media/kamilyassis/pln/refs/heads/main/2026-03-29_label_datasets/data/out/conexao-labels-full-4.1nano-p2.csv"
local_dataset_path = "data/out/semeval-treated.csv"
online_dataset_path = "https://media.githubusercontent.com/media/kamilyassis/pln/refs/heads/main/2026-03-30_bert/data/out/semeval-treated.csv"

in_colab = True

DATASET_ARGS = {
    "filepath_or_buffer": local_dataset_path if not in_colab else online_dataset_path,
    "sep": '|'
}

Using device: cuda


In [4]:

# ──────────────────────────────────────────────
# 2. DATASET
# ──────────────────────────────────────────────
class TextClassificationDataset(Dataset):
    """
    Generic dataset for text classification.

    Args:
        texts  : list[str]  – raw input sentences
        labels : list[int]  – integer class indices (0 … NUM_LABELS-1)
        tokenizer           – HuggingFace tokenizer
        max_len : int       – max sequence length
    """

    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }


In [ ]:


# ──────────────────────────────────────────────
# 3. METRICS
# ──────────────────────────────────────────────
def compute_metrics(preds, labels):
    preds = np.argmax(preds, axis=1)
    acc   = accuracy_score(labels, preds)
    report = classification_report(labels, preds, zero_division=0)
    return acc, report


# ──────────────────────────────────────────────
# 4. TRAIN ONE EPOCH
# ──────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0.0

    for batch in tqdm(loader, desc="Training batches"):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)


# ──────────────────────────────────────────────
# 5. EVALUATE
# ──────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            total_loss += outputs.loss.item()
            all_preds.append(outputs.logits.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_preds = np.vstack(all_preds)
    acc, report = compute_metrics(all_preds, all_labels)
    return total_loss / len(loader), acc, report


# ──────────────────────────────────────────────
# 6. INFERENCE / PREDICT
# ──────────────────────────────────────────────
def predict(texts, model, tokenizer, id2label=None):
    """
    Run inference on a list of strings.

    Returns:
        predicted class indices (and labels if id2label is provided)
    """
    model.eval()
    all_preds = []

    dataset = TextClassificationDataset(
        texts, [0] * len(texts), tokenizer  # dummy labels
    )
    loader = DataLoader(dataset, batch_size=BATCH_SIZE)

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)

            outputs  = model(input_ids=input_ids, attention_mask=attention_mask)
            preds    = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)

    if id2label:
        return [id2label[p] for p in all_preds]
    return all_preds



: 

In [6]:

# ── 7a. Load your data ──────────────────────
# Replace this block with your actual data loading logic.
# `texts`  → list of raw strings
# `labels` → list of int class indices (0-based)

df = pd.read_csv(**DATASET_ARGS)
texts  = df['sentence'].tolist()  # or 'sentence' if that's your column
labels = df['label'].tolist() # make sure these are integers from 0 to NUM_LABELS-1

# Optionally define a human-readable mapping (used at inference time)
id2label = {
    0: "No_Label",
    1: "Loaded_Language",
    2: "Name_Calling-Labeling",
    3: "Doubt",
    4: "Repetition",
    5: "Appeal_to_Fear-Prejudice",
    6: "Flag_Waving",
    7: "Exaggeration-Minimisation"
}
label2id = {v: k for k, v in id2label.items()}

labels = [label2id[l] for l in labels]  # convert to integer indices if needed

from imblearn.over_sampling import RandomOverSampler

# Create 2D array for texts (required by imbalanced-learn)
texts_array = np.array(texts).reshape(-1, 1)

# Oversample to balance all classes
ros = RandomOverSampler(random_state=SEED)
texts_resampled, labels_resampled = ros.fit_resample(texts_array, labels)

# Garantir conversão segura
texts = [str(t).strip() for t in texts_resampled.squeeze()] if isinstance(texts_resampled, np.ndarray) else texts_resampled
labels = list(labels_resampled) if not isinstance(labels_resampled, list) else labels_resampled

# ── 7b. Train / val / test split ────────────
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=SEED, stratify=labels
)

: 

: 

In [ ]:
train_texts[:5]

['se voce conseguir prender a respiracao ate que o ponto vermelho se mova de a para b, voce esta livre de covid 19 no momento. teste simples e cobicoso. teste gratis sem nenhum custo. ajude a salvar uma vida. espere ate que o ponto vermelho se mova para a antes de comecar a prender a respiracao. repassando.',
 'eleitores de lula não precisarão voltar às urnas no 2° turno, diz moraes. presidente do tse deu informação durante sessão desta quinta-feira (6). moraes tem reiterado que nova tecnologia do tse permite reaproveitar o voto do 1° turno. o ministro alexandre de moraes, presidente do tribunal superior eleitoral (tse), informou que graças as novas tecnologias implementadas nas urnas eletrônicas, quem já votou no 1º turno em lula não precisará retornar para votar no 2º turno. a tecnologia foi apresentada pelo equipe de ti do tse e poupará milhões de eleitores da necessidade de retornar às umas em 3/outubro. “nossa urna é de última geração e quem votou na chapa lula “juntos pelo brasil

In [ ]:

# ── 7c. Tokenizer & model ───────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
).to(DEVICE)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect i

In [ ]:

# ── 7d. DataLoaders ─────────────────────────
train_dataset = TextClassificationDataset(train_texts, train_labels, tokenizer)
val_dataset   = TextClassificationDataset(val_texts,   val_labels,   tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)


In [ ]:

# ── 7e. Optimizer & scheduler ───────────────
total_steps   = len(train_loader) * EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)

optimizer = AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)


In [ ]:

# ── 7f. Training loop ───────────────────────
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, scheduler)
    val_loss, val_acc, val_report = evaluate(model, val_loader)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )
    print(val_report)

    # Save best checkpoint
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        os.makedirs(SAVE_DIR, exist_ok=True)
        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        # save model to google drive from colab
        from google.colab import drive
        drive.mount("/content/drive")
        model.save_pretrained(f"/content/drive/MyDrive/bert_tiny_classifier_semeval/epoch_{epoch}")
        tokenizer.save_pretrained(f"/content/drive/MyDrive/bert_tiny_classifier_semeval/epoch_{epoch}")
        print(f"  ✓ New best model saved to {SAVE_DIR}")

print(f"\nTraining complete. Best val accuracy: {best_val_acc:.4f}")


Training batches: 100%|██████████| 142/142 [00:11<00:00, 12.16it/s]


Epoch 1/20 | Train Loss: 2.0881 | Val Loss: 2.0706 | Val Acc: 0.1584
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       141
           1       0.08      0.13      0.10       141
           2       0.11      0.07      0.09       141
           3       0.18      0.02      0.04       142
           4       0.18      0.35      0.24       141
           5       0.14      0.09      0.11       141
           6       0.24      0.05      0.08       142
           7       0.19      0.55      0.28       141

    accuracy                           0.16      1130
   macro avg       0.14      0.16      0.12      1130
weighted avg       0.14      0.16      0.12      1130



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ New best model saved to ./bert_tiny_classifier


Training batches: 100%|██████████| 142/142 [00:11<00:00, 12.49it/s]


Epoch 20/20 | Train Loss: 0.4124 | Val Loss: 0.4674 | Val Acc: 0.8708
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       141
           1       0.70      0.53      0.60       141
           2       0.77      0.87      0.81       141
           3       0.94      1.00      0.97       142
           4       0.94      0.95      0.95       141
           5       0.69      0.62      0.65       141
           6       0.93      1.00      0.96       142
           7       0.96      1.00      0.98       141

    accuracy                           0.87      1130
   macro avg       0.86      0.87      0.86      1130
weighted avg       0.86      0.87      0.86      1130


Training complete. Best val accuracy: 0.8717


In [ ]:

# ── 7g. Inference example ───────────────────
best_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to(DEVICE)
sample     = ["Você prefere o brasil ou cuba?", "Caralho porra que merda de politica do caralho"]
preds      = predict(sample, best_model, tokenizer, id2label=id2label)
for text, pred in zip(sample, preds):
    print(f"  [{pred}] {text}")

Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

  [Doubt] Você prefere o brasil ou cuba?
  [Name_Calling-Labeling] Caralho porra que merda de politica do caralho
